<a href="https://colab.research.google.com/github/AishwaryaPapineni/MediRisk-AI/blob/main/medirisk_cl_6_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# MEDIRISK - FAST FAERS MACHINE LEARNING PROJECT
# ============================================================

import pandas as pd
import numpy as np
import os
import joblib
import zipfile

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. FIND UPLOADED FILES
# ------------------------------------------------------------

# Extract zip file if present
zip_file = None
for f in os.listdir("/content"):
    if f.endswith(".zip"):
        zip_file = os.path.join("/content", f)
        break

if zip_file:
    print(f"Unzipping {zip_file}...")
    with zipfile.ZipFile(zip_file, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Unzipping complete.")

# After unzipping, check if 'ASCII' directory exists and use it
unzipped_data_path = "/content/"
if "ASCII" in os.listdir("/content") and os.path.isdir(os.path.join("/content", "ASCII")):
    unzipped_data_path = "/content/ASCII/"
    print(f"Detected 'ASCII' subdirectory, looking for files in {unzipped_data_path}")

all_files = os.listdir(unzipped_data_path)

def find_file(prefix):
    for f in all_files:
        # Prioritize .txt files over .pdf files
        if f.upper().startswith(prefix) and f.upper().endswith('.TXT'):
            return unzipped_data_path + f
    # Fallback to general prefix if no .txt found (though not expected for this dataset)
    for f in all_files:
        if f.upper().startswith(prefix):
            return unzipped_data_path + f
    return None

demo_file = find_file("DEMO")
drug_file = find_file("DRUG")
reac_file = find_file("REAC")
outc_file = find_file("OUTC")

print("Files found:")
print("DEMO:", demo_file)
print("DRUG:", drug_file)
print("REAC:", reac_file)
print("OUTC:", outc_file)


# ------------------------------------------------------------
# 2. LOAD ONLY A SAMPLE
# ------------------------------------------------------------

print("\nLoading data...")

demo = pd.read_csv(
    demo_file,
    sep="$",
    encoding="latin1",
    nrows=30000
)

drug = pd.read_csv(
    drug_file,
    sep="$",
    encoding="latin1",
    nrows=50000
)

reac = pd.read_csv(
    reac_file,
    sep="$",
    encoding="latin1",
    nrows=50000
)

outc = pd.read_csv(
    outc_file,
    sep="$",
    encoding="latin1",
    nrows=50000
)

print("DEMO:", demo.shape)
print("DRUG:", drug.shape)
print("REAC:", reac.shape)
print("OUTC:", outc.shape)


# ------------------------------------------------------------
# 3. CLEAN COLUMN NAMES
# ------------------------------------------------------------

for df in [demo, drug, reac, outc]:
    df.columns = (
        df.columns
        .str.strip()
        .str.upper()
    )


# ------------------------------------------------------------
# 4. PRIMARYID
# ------------------------------------------------------------

for df in [demo, drug, reac, outc]:

    df["PRIMARYID"] = pd.to_numeric(
        df["PRIMARYID"],
        errors="coerce"
    )

    df.dropna(
        subset=["PRIMARYID"],
        inplace=True
    )

    df["PRIMARYID"] = df["PRIMARYID"].astype(int)


# ------------------------------------------------------------
# 5. DEMOGRAPHICS
# ------------------------------------------------------------

demo["AGE"] = pd.to_numeric(
    demo["AGE"],
    errors="coerce"
)

demo.loc[
    (demo["AGE"] < 0) |
    (demo["AGE"] > 120),
    "AGE"
] = np.nan

demo["SEX"] = (
    demo["SEX"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# One row per report
demo = demo.drop_duplicates(
    "PRIMARYID"
)


# ------------------------------------------------------------
# 6. DRUG FEATURES
# ------------------------------------------------------------

drug["DRUGNAME"] = (
    drug["DRUGNAME"]
    .astype(str)
    .str.upper()
    .str.strip()
)

drug_count = (
    drug
    .groupby("PRIMARYID")
    .size()
    .reset_index(
        name="num_drugs"
    )
)

# Suspect drug count
if "ROLE_COD" in drug.columns:

    drug["ROLE_COD"] = (
        drug["ROLE_COD"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    suspect_count = (
        drug[
            drug["ROLE_COD"] == "PS"
        ]
        .groupby("PRIMARYID")
        .size()
        .reset_index(
            name="num_suspect_drugs"
        )
    )

else:

    suspect_count = pd.DataFrame(
        columns=[
            "PRIMARYID",
            "num_suspect_drugs"
        ]
    )

drug_features = drug_count.merge(
    suspect_count,
    on="PRIMARYID",
    how="left"
)

drug_features[
    "num_suspect_drugs"
] = drug_features[
    "num_suspect_drugs"
].fillna(0)


# ------------------------------------------------------------
# 7. REACTION FEATURES
# ------------------------------------------------------------

if "PT" in reac.columns:
    reaction_column = "PT"
else:
    reaction_column = [
        c for c in reac.columns
        if "PT" in c
    ][0]

reaction_count = (
    reac
    .groupby("PRIMARYID")
    .size()
    .reset_index(
        name="num_reactions"
    )
)


# ------------------------------------------------------------
# 8. OUTCOME / TARGET
# ------------------------------------------------------------

outc["OUTC_COD"] = (
    outc["OUTC_COD"]
    .astype(str)
    .str.upper()
    .str.strip()
)

# Serious outcome codes
serious_codes = {
    "DE",   # Death
    "LT",   # Life-threatening
    "HO",   # Hospitalization
    "DS",   # Disability
    "CA",   # Congenital anomaly
    "RI"    # Required intervention
}

outc["serious_event"] = (
    outc["OUTC_COD"]
    .isin(serious_codes)
    .astype(int)
)

outcome_features = (
    outc
    .groupby("PRIMARYID")["serious_event"]
    .max()
    .reset_index()
)


# ------------------------------------------------------------
# 9. MERGE ALL TABLES
# ------------------------------------------------------------

print("\nCombining tables...")

df = demo.merge(
    drug_features,
    on="PRIMARYID",
    how="left"
)

df = df.merge(
    reaction_count,
    on="PRIMARYID",
    how="left"
)

df = df.merge(
    outcome_features,
    on="PRIMARYID",
    how="left"
)


# ------------------------------------------------------------
# 10. CLEAN FINAL DATASET
# ------------------------------------------------------------

df = df.drop_duplicates(
    "PRIMARYID"
)

# Numeric columns
for col in [
    "AGE",
    "num_drugs",
    "num_suspect_drugs",
    "num_reactions"
]:

    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# Categorical column
df["SEX"] = (
    df["SEX"]
    .fillna("UNKNOWN")
    .astype(str)
)

# Remove reports without outcome information
df = df.dropna(
    subset=["serious_event"]
)


# ------------------------------------------------------------
# 11. CHECK DATA
# ------------------------------------------------------------

print("\nFinal dataset:")
print(df.shape)

print("\nTarget distribution:")
print(
    df["serious_event"]
    .value_counts()
)

print("\nFirst 5 rows:")
display(df.head())


# ------------------------------------------------------------
# 12. SAVE CLEAN DATA
# ------------------------------------------------------------

df.to_csv(
    "medrisk_clean.csv",
    index=False
)


# ------------------------------------------------------------
# 13. MACHINE LEARNING FEATURES
# ------------------------------------------------------------

features = [
    "AGE",
    "num_drugs",
    "num_suspect_drugs",
    "num_reactions",
    "SEX"
]

X = df[features]

y = df["serious_event"]


# ------------------------------------------------------------
# 14. TRAIN TEST SPLIT
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ------------------------------------------------------------
# 15. PREPROCESSING
# ------------------------------------------------------------

numeric_features = [
    "AGE",
    "num_drugs",
    "num_suspect_drugs",
    "num_reactions"
]

categorical_features = [
    "SEX"
]

preprocessor = ColumnTransformer(
    transformers=[

        (
            "numeric",
            SimpleImputer(
                strategy="median"
            ),
            numeric_features
        ),

        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "encoder",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 16. LOGISTIC REGRESSION
# ------------------------------------------------------------

logistic = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),

    (
        "model",
        LogisticRegression(
            max_iter=300,
            class_weight="balanced"
        )
    )
])

print("\nTraining Logistic Regression...")

logistic.fit(
    X_train,
    y_train
)

logistic_prob = logistic.predict_proba(
    X_test
)[:, 1]

logistic_pred = (
    logistic_prob >= 0.5
).astype(int)

logistic_auc = roc_auc_score(
    y_test,
    logistic_prob
)


# ------------------------------------------------------------
# 17. RANDOM FOREST
# ------------------------------------------------------------

rf = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),

    (
        "model",
        RandomForestClassifier(
            n_estimators=50,
            max_depth=10,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])

print("Training Random Forest...")

rf.fit(
    X_train,
    y_train
)

rf_prob = rf.predict_proba(
    X_test
)[:, 1]

rf_pred = (
    rf_prob >= 0.5
).astype(int)

rf_auc = roc_auc_score(
    y_test,
    rf_prob
)


# ------------------------------------------------------------
# 18. RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MEDIRISK RESULTS")
print("=" * 60)

print(
    "\nLogistic Regression ROC-AUC:",
    round(logistic_auc, 4)
)

print(
    "\nRandom Forest ROC-AUC:",
    round(rf_auc, 4)
)

print("\nRandom Forest Classification Report:")

print(
    classification_report(
        y_test,
        rf_pred
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        rf_pred
    )
)


# ------------------------------------------------------------
# 19. SAVE MODEL
# ------------------------------------------------------------

joblib.dump(
    rf,
    "medrisk_random_forest.pkl"
)


# ------------------------------------------------------------
# 20. SAVE RESULTS
# ------------------------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],

    "ROC_AUC": [
        logistic_auc,
        rf_auc
    ]
})

results.to_csv(
    "model_results.csv",
    index=False
)


# ------------------------------------------------------------
# 21. PROJECT SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("PROJECT COMPLETED")
print("=" * 60)

print("""
Created files:

1. medrisk_clean.csv
2. model_results.csv
3. medrisk_random_forest.pkl

Project pipeline:

FAERS
  ↓
Data Cleaning
  ↓
DEMO + DRUG + REAC + OUTC
  ↓
Feature Engineering
  ↓
Serious Event Prediction
  ↓
Logistic Regression
  ↓
Random Forest
  ↓
Model Evaluation
""")

Unzipping /content/faers_ascii_2026q2.zip...
Unzipping complete.
Detected 'ASCII' subdirectory, looking for files in /content/ASCII/
Files found:
DEMO: /content/ASCII/DEMO26Q2.txt
DRUG: /content/ASCII/DRUG26Q2.txt
REAC: /content/ASCII/REAC26Q2.txt
OUTC: /content/ASCII/OUTC26Q2.txt

Loading data...
DEMO: (30000, 25)
DRUG: (50000, 20)
REAC: (50000, 4)
OUTC: (50000, 3)

Combining tables...

Final dataset:
(21685, 29)

Target distribution:
serious_event
1.0    11704
0.0     9981
Name: count, dtype: int64

First 5 rows:


,PRIMARYID,CASEID,CASEVERSION,I_F_CODE,EVENT_DT,MFR_DT,INIT_FDA_DT,FDA_DT,REPT_COD,AUTH_NUM,...,WT_COD,REPT_DT,TO_MFR,OCCP_COD,REPORTER_COUNTRY,OCCR_COUNTRY,num_drugs,num_suspect_drugs,num_reactions,serious_event
0,100140224,10014022,4,F,NaN,20260508,20140317,20260513,EXP,NaN,...,KG,20260513,NaN,MD,EU,EU,7.0,1.0,1.0,0.0
1,100218353,10021835,3,F,NaN,20260528,20140319,20260603,EXP,NaN,...,NaN,20260603,NaN,LW,US,US,6.0,1.0,4.0,0.0
2,101009832,10100983,2,F,NaN,20140402,20140424,20260624,EXP,NaN,...,NaN,20260624,NaN,MD,EU,EU,3.0,1.0,2.0,1.0
39,101448952,10144895,2,F,NaN,20130905,20140430,20260629,PER,NaN,...,NaN,20260628,NaN,NaN,US,NaN,1.0,1.0,1.0,0.0
70,101754163,10175416,3,F,NaN,20260526,20140515,20260529,EXP,NaN,...,KG,20260529,NaN,MD,EU,EU,8.0,1.0,2.0,1.0



Training Logistic Regression...
Training Random Forest...

MEDIRISK RESULTS

Logistic Regression ROC-AUC: 0.5706

Random Forest ROC-AUC: 0.605

Random Forest Classification Report:
              precision    recall  f1-score   support

         0.0       0.53      0.55      0.54      1996
         1.0       0.60      0.58      0.59      2341

    accuracy                           0.57      4337
   macro avg       0.57      0.57      0.57      4337
weighted avg       0.57      0.57      0.57      4337


Confusion Matrix:
[[1104  892]
 [ 981 1360]]

PROJECT COMPLETED

Created files:

1. medrisk_clean.csv
2. model_results.csv
3. medrisk_random_forest.pkl

Project pipeline:

FAERS
  ↓
Data Cleaning
  ↓
DEMO + DRUG + REAC + OUTC
  ↓
Feature Engineering
  ↓
Serious Event Prediction
  ↓
Logistic Regression
  ↓
Random Forest
  ↓
Model Evaluation

